In [ ]:
! pip install matplotlib seaborn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.simplefilter('ignore')

In [ ]:
df = pd.read_csv('train.csv')
df.head()

In [ ]:
df.drop(columns=["Unnamed: 0","Booking_ID"],inplace=True)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
cat_col = ['type_of_meal_plan', 'required_car_parking_space',
       'room_type_reserved', 'market_segment_type', 'repeated_guest',
    'booking_status'
]

In [ ]:
num_col = ['no_of_adults', 'no_of_children', 'no_of_weekend_nights',
       'no_of_week_nights','lead_time', 'arrival_year', 'arrival_month',
       'arrival_date','no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled',
       'avg_price_per_room', 'no_of_special_requests']

In [ ]:
df.info()

### Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

mappings = {}

for col in cat_col:
    df[col] = label_encoder.fit_transform(df[col])

    mappings[col]= {label:code for label,code in zip(label_encoder.classes_,label_encoder.transform(label_encoder.classes_))}

In [ ]:
mappings

In [ ]:
##Multicolinarity

In [ ]:
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

X = add_constant(df)

vif_data = pd.DataFrame()
vif_data['features'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

In [ ]:
corr = df.corr()

In [ ]:
corr

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(corr)

In [ ]:
skewness = df.skew()
skewness

In [ ]:
for col in df.columns:
    if skewness[col]>5:
        df[col] = np.log1p(df[col])

In [ ]:
df[col]

In [ ]:
skewness = df.skew()
skewness

In [ ]:
df['booking_status'].value_counts()

In [ ]:
X = df.drop(columns=['booking_status'])
y = df['booking_status']

In [ ]:
X.head()

In [ ]:
y

In [ ]:
!pip install imbalanced-learn

In [ ]:
y.value_counts()

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(X,y)

In [ ]:
y_res.value_counts()

In [ ]:
balanced_df = pd.DataFrame(x_res,columns=X.columns)
balanced_df['booking_status'] = y_res

In [ ]:
balanced_df.shape

In [ ]:
df = balanced_df.copy()

In [ ]:
df.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['booking_status'])
y = df['booking_status']

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X,y)

In [ ]:
model.feature_importances_

In [ ]:
feature_importance = model.feature_importances_

In [ ]:
feature_importance_df = pd.DataFrame(
    {'feature': X.columns,
    'importance': feature_importance
})

In [ ]:
top_features = feature_importance_df.sort_values(by='importance',ascending=False)

In [ ]:
top_10_features = top_features['feature'].head(10).values
# top_10_df = df

In [ ]:
top_10_df = df[top_10_features.tolist()+ ['booking_status']]

In [ ]:
top_10_df.shape

In [ ]:
df = top_10_df.copy()

In [ ]:
!pip install xgboost lightgbm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

In [ ]:
X = df.drop(columns=['booking_status'])
y = df['booking_status']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [ ]:
classifiers = {
    'Random forest': RandomForestClassifier(random_state=42),
    'LogisticRegression': LogisticRegression(random_state=42),
    'Graient Boosting' : GradientBoostingClassifier(random_state=42),
    'Support vector classifier': SVC(random_state=42),
    'Decision tree' : DecisionTreeClassifier(random_state=42),
    'KNN' : KNeighborsClassifier(),
    'Naive bayes': GaussianNB(),
    'XGboost': XGBClassifier(random_state=42),
    'Adaboost': AdaBoostClassifier(random_state=42),
    'LGBM': LGBMClassifier(random_state=42)
}

In [ ]:
metrics = {
    'Model' :[],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1': []

}

In [ ]:
for model_name, classifier in classifiers.items():
    classifier.fit(X_train,y_train)

    y_pred = classifier.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)


    metrics['Model'].append(model_name)
    metrics['Accuracy'].append(accuracy)
    metrics['Recall'].append(recall)
    metrics['F1'].append(f1)
    metrics['Precision'].append(precision)



In [ ]:
metrics_df = pd.DataFrame(metrics)

In [ ]:
metrics_df

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [ ]:
X = df.drop(columns=['booking_status'])
y = df['booking_status']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [ ]:
rf = RandomForestClassifier(random_state=42)

In [ ]:
params = {
    'n_estimators': randint(100,500) ,
    'max_depth' : randint(10,50),
    'min_samples_split': randint(2,10),
    'min_samples_leaf': randint(1,5),
    'bootstrap' :[True,False],

}

In [ ]:
random_search = RandomizedSearchCV(estimator=rf,
                                   param_distributions=params,
                                   n_iter=5,
                                   cv =2,
                                   verbose=2,
                                   random_state=42,
                                   scoring='accuracy')

In [ ]:
random_search.fit(X_train,y_train)

In [ ]:
random_search.best_params_

In [ ]:
best_rf_model = random_search.best_estimator_

In [ ]:
y_pred  = best_rf_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

In [ ]:
accuracy

In [ ]:
recall

In [ ]:
precision

In [ ]:
f1

In [ ]:
import joblib

joblib.dump(best_rf_model, "random_forest.pkl")

In [ ]:
loaded_model = joblib.load("random_forest.pkl")

In [ ]:
loaded_model.predict(np.array(X_train[0:1]).reshape(1,-1))